# 🎙️ Máy chủ giọng nói GPT-SoVITS cho RP Chat (giọng vita)
**Cách chạy:** `Runtime → Change runtime type → T4 GPU`, rồi bấm **`Runtime → Run all`**. Lần đầu cài mất khoảng 10–20 phút. Khi xong, cuối ô 3 sẽ hiện:
- 🌐 **Link máy chủ TTS** (dạng `https://….trycloudflare.com`) → dán vào RP Chat › Cài đặt › Giọng nói.
- 📱 **Link mở RP Chat tự điền sẵn** → bấm trên điện thoại là xong.

Giữ tab này mở trong lúc chat. Link đổi mỗi lần chạy lại. Dùng xong: `Runtime → Disconnect and delete runtime`.

**Ngôn ngữ:** GPT-SoVITS chỉ hỗ trợ **Nhật (ja), Anh (en), Trung (zh), Hàn (ko), Quảng Đông (yue)**. Không có tiếng Việt: câu tiếng Việt sẽ được bỏ dấu và đọc bằng âm tiếng Anh (lơ lớ). Giọng vita train từ audio tiếng Nhật nên đọc **tiếng Nhật** hay nhất.

API: `GET /health`, `GET /voices`, `POST /tts` JSON `{"text","voice","lang","speed","format"}` → `audio/wav`.


In [ ]:
#@title 1. GPU + Google Drive + chọn giọng
VOICE_DIR = "/content/drive/MyDrive/vita-voice-final"  #@param {type:"string"}
#@markdown Lời của đoạn audio mẫu `ref.wav` (để trống = tự tìm trong file .list / .txt, không có thì tự nghe bằng Whisper):
REF_TEXT = ""  #@param {type:"string"}
REF_LANG = "ja"  #@param ["ja","en","zh","ko","yue"]
#@markdown Thư mục giọng phụ (tùy chọn): mỗi thư mục con = 1 giọng, chứa `ref.wav` + `ref.txt` (+ `lang.txt`, + file .ckpt/.pth riêng nếu có):
EXTRA_VOICES_DIR = "/content/drive/MyDrive/rp-voices"  #@param {type:"string"}
import subprocess, os
out = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout
GPU = "NVIDIA" in out
print(out if GPU else "⚠️ Không có GPU: vẫn chạy được trên CPU nhưng mỗi câu mất 10-30 giây. Nên chọn T4 GPU.")
from google.colab import drive
drive.mount("/content/drive")
W = f"{VOICE_DIR}/weights"
assert os.path.isdir(W), f"❌ Không thấy {W}. Kiểm tra lại thư mục model vita trong Drive."
print("✅ weights:", os.listdir(W))
print("✅ files:", [f for f in os.listdir(VOICE_DIR) if not os.path.isdir(f"{VOICE_DIR}/{f}")])


In [ ]:
#@title 2. Cài GPT-SoVITS (~10-20 phút lần đầu)
import os, subprocess
dev = "CU126" if GPU else "CPU"
setup = f"""
set -e
cd /content
if [ ! -x /content/conda/bin/conda ]; then
  wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
  bash /tmp/mf.sh -b -p /content/conda
fi
source /content/conda/bin/activate
[ -d GPT-SoVITS ] || git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
cd GPT-SoVITS
conda env list | awk '{{print $1}}' | grep -Fxq GPTSoVITS || conda create -n GPTSoVITS python=3.10 -y -q
conda activate GPTSoVITS
[ -f /content/.gsv_installed ] || bash install.sh --device {dev} --source HF
"""
open("/content/setup.sh","w").write(setup)
subprocess.run("bash /content/setup.sh 2>&1 | tail -n 25", shell=True, executable="/bin/bash")
assert os.path.isdir("/content/GPT-SoVITS/GPT_SoVITS/pretrained_models"), "❌ Cài lỗi, xem log phía trên."
PIP = "/content/conda/envs/GPTSoVITS/bin/pip"
if GPU and not os.path.exists("/content/.gsv_installed"):  # sửa lệch CUDA giữa torch và torchaudio (đã dùng ổn ở notebook vita)
    subprocess.run(f"{PIP} install -q torch==2.11.0 torchaudio==2.11.0 torchvision --index-url https://download.pytorch.org/whl/cu126", shell=True)
# resampy: GPT-SoVITS cần nhưng requirements.txt (Linux) thiếu; faster-whisper: tự nghe lời audio mẫu nếu chưa có
subprocess.run(f"{PIP} install -q 'resampy>=0.4.2,<0.5' faster-whisper soundfile", shell=True)
open("/content/.gsv_installed","w").write("ok")
print("✅ Cài xong")


In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"  # fix Colab matplotlib_inline backend error
#@title 3. Chạy máy chủ giọng nói + link (giữ ô này chạy)
import os, re, json, glob, time, shutil, subprocess, urllib.request
os.chdir("/content/GPT-SoVITS")
PY = "/content/conda/envs/GPTSoVITS/bin/python"
APP_URL = "https://hoangnguyencccc194-blip.github.io/rpchat-cd8e0282/"
open("rp_tts_server.py", "w", encoding="utf-8").write('# RP Chat TTS server: a small FastAPI wrapper around GPT-SoVITS (TTS_infer_pack).\n# Run from the GPT-SoVITS repo root:  python rp_tts_server.py --voices voices.json --port 9880\n# API:  GET /health   GET /voices   GET|POST /tts  (returns audio/wav or audio/ogg)\nimport os, sys, io, re, json, time, argparse, threading, traceback\nnow_dir = os.getcwd()\nsys.path.insert(0, now_dir); sys.path.insert(0, os.path.join(now_dir, "GPT_SoVITS"))\nos.environ.setdefault("MPLBACKEND", "Agg")\n\nimport numpy as np, soundfile as sf, torch, torchaudio\n\n# torchaudio>=2.9 needs torchcodec for load(); use soundfile instead so a torch/torchcodec mismatch can\'t break us.\ndef _sf_load(path, *a, **k):\n    d, sr = sf.read(path, dtype="float32", always_2d=True)\n    return torch.from_numpy(np.ascontiguousarray(d.T)), sr\ntorchaudio.load = _sf_load\n\nfrom fastapi import FastAPI, Request\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import JSONResponse, Response\nfrom starlette.concurrency import run_in_threadpool\nimport uvicorn\nfrom GPT_SoVITS.TTS_infer_pack.TTS import TTS, TTS_Config\n\nap = argparse.ArgumentParser()\nap.add_argument("--voices", default="voices.json")\nap.add_argument("--host", default="127.0.0.1")\nap.add_argument("--port", type=int, default=9880)\nap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\nap.add_argument("--half", default="auto")\nap.add_argument("--max-chars", type=int, default=400)\nargs = ap.parse_args()\n\nCFG = json.load(open(args.voices, encoding="utf-8"))\nVOICES = CFG["voices"]\nDEFAULT_VOICE = CFG.get("default") or next(iter(VOICES))\nhalf = (args.device.startswith("cuda")) if args.half == "auto" else args.half.lower() in ("1", "true", "yes")\nv0 = VOICES[DEFAULT_VOICE]\nPM = "GPT_SoVITS/pretrained_models"\ntts_config = TTS_Config({"custom": {\n    "device": args.device, "is_half": half, "version": CFG.get("version", "v2Pro"),\n    "t2s_weights_path": v0["gpt"], "vits_weights_path": v0["sovits"],\n    "bert_base_path": f"{PM}/chinese-roberta-wwm-ext-large", "cnhuhbert_base_path": f"{PM}/chinese-hubert-base"}})\npipe = TTS(tts_config)\nLOADED = {"gpt": v0["gpt"], "sovits": v0["sovits"]}\nLOCK = threading.Lock()\nSTATS = {"requests": 0, "errors": 0, "started": time.time(), "last_ms": None}\nLANGS = list(tts_config.languages)\nprint(f"[rp-tts] version={tts_config.version} device={tts_config.device} half={tts_config.is_half} langs={LANGS}", flush=True)\n\nALLOW_RE = os.environ.get("CORS_ORIGIN_REGEX",\n    r"https://hoangnguyencccc194-blip\\.github\\.io|https?://(localhost|127\\.0\\.0\\.1)(:\\d+)?")\napp = FastAPI(title="RP Chat TTS (GPT-SoVITS)")\napp.add_middleware(CORSMiddleware, allow_origin_regex=ALLOW_RE if os.environ.get("CORS_ALLOW_ALL") != "1" else ".*",\n                   allow_methods=["GET", "POST", "OPTIONS"], allow_headers=["*"],\n                   expose_headers=["X-Voice", "X-Lang", "X-Duration", "X-Gen-Ms", "X-Text"], max_age=86400)\n\ndef clean_text(t: str) -> str:\n    t = str(t or "")\n    t = re.sub(r"\\(\\(.*?\\)\\)|\\[\\s*OOC[^\\]]*\\]|\\(\\s*OOC[^)]*\\)|^\\s*OOC\\b.*$", " ", t, flags=re.I | re.M | re.S)\n    t = re.sub(r"\\*[^*\\n]+\\*", " ", t)          # *action*\n    t = re.sub(r"https?://\\S+", " ", t)\n    t = re.sub(r"[*_~`#>|]", " ", t)\n    t = re.sub(r"[\\U0001F000-\\U0001FAFF\\u2600-\\u27BF\\uFE0F]", " ", t)  # emoji\n    return re.sub(r"\\s+", " ", t).strip()\n\nVI_CHARS = re.compile(r"[ăâđêôơưàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵ]", re.I)\ndef latinize_vi(t: str) -> str:\n    import unicodedata\n    t = t.replace("đ", "d").replace("Đ", "D")\n    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")\n\ndef map_lang(l: str, v: dict) -> str:\n    l = (l or "auto").strip().lower()\n    alias = {"jp": "ja", "japanese": "ja", "english": "en", "chinese": "zh", "cn": "zh", "korean": "ko", "kr": "ko", "cantonese": "yue"}\n    l = alias.get(l, l)\n    return l if l in LANGS else "auto"          # e.g. "vi" is not supported by GPT-SoVITS -> auto\n\ndef ensure_weights(v):\n    if v["gpt"] != LOADED["gpt"]:\n        pipe.init_t2s_weights(v["gpt"]); LOADED["gpt"] = v["gpt"]\n    if v["sovits"] != LOADED["sovits"]:\n        pipe.init_vits_weights(v["sovits"]); LOADED["sovits"] = v["sovits"]\n\ndef pack(audio: np.ndarray, sr: int, fmt: str) -> bytes:\n    buf = io.BytesIO()\n    if audio.dtype != np.int16:\n        audio = np.clip(audio, -1, 1) if audio.dtype.kind == "f" else audio\n        audio = (audio * 32767).astype(np.int16) if audio.dtype.kind == "f" else audio.astype(np.int16)\n    if fmt == "ogg": sf.write(buf, audio, sr, format="OGG", subtype="VORBIS")\n    else: sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")\n    return buf.getvalue()\n\ndef synth(text, voice, lang, speed, fmt, seed):\n    v = VOICES[voice]\n    with LOCK:\n        ensure_weights(v)\n        t0 = time.time()\n        req = {"text": text, "text_lang": lang, "ref_audio_path": v["ref"], "prompt_text": v.get("text", ""),\n               "prompt_lang": v.get("lang", "ja"), "top_k": 15, "top_p": 1, "temperature": 1,\n               "text_split_method": "cut5", "batch_size": 1, "speed_factor": float(speed),\n               "fragment_interval": 0.25, "seed": int(seed), "parallel_infer": True,\n               "repetition_penalty": 1.35, "split_bucket": False}\n        chunks, sr = [], 32000\n        for sr, a in pipe.run(req): chunks.append(a)\n        audio = np.concatenate(chunks) if chunks else np.zeros(1, np.int16)\n        ms = int((time.time() - t0) * 1000)\n    return pack(audio, sr, fmt), sr, len(audio) / sr, ms\n\n@app.get("/health")\n@app.get("/")\ndef health():\n    return {"ok": True, "service": "rp-tts", "engine": "GPT-SoVITS", "model_version": tts_config.version,\n            "device": str(tts_config.device), "half": bool(tts_config.is_half), "languages": LANGS,\n            "voices": list(VOICES), "default_voice": DEFAULT_VOICE, "busy": LOCK.locked(),\n            "formats": ["wav", "ogg"], "max_chars": args.max_chars,\n            "uptime_s": int(time.time() - STATS["started"]), "requests": STATS["requests"], "last_ms": STATS["last_ms"]}\n\n@app.get("/voices")\ndef voices():\n    return {"default": DEFAULT_VOICE, "voices": [{"id": k, "name": v.get("name", k), "prompt_lang": v.get("lang", "ja"),\n            "has_prompt_text": bool(v.get("text"))} for k, v in VOICES.items()]}\n\n@app.api_route("/tts", methods=["GET", "POST"])\nasync def tts(request: Request):\n    p = dict(request.query_params)\n    if request.method == "POST":\n        try:\n            body = await request.json()\n            if isinstance(body, dict): p.update(body)\n        except Exception:\n            return JSONResponse({"ok": False, "error": "body must be JSON"}, 400)\n    text = clean_text(p.get("text", ""))[: args.max_chars]\n    if not re.search(r"\\w", text): return JSONResponse({"ok": False, "error": "empty text after cleaning"}, 400)\n    voice = p.get("voice") or DEFAULT_VOICE\n    if voice not in VOICES: voice = DEFAULT_VOICE\n    raw_lang = str(p.get("lang") or p.get("text_lang") or "auto").lower()\n    lang = map_lang(raw_lang, VOICES[voice])\n    note = ""\n    # GPT-SoVITS has no Vietnamese front-end: Vietnamese text is dropped or crashes. Strip the tone marks and\n    # read it with the English front-end instead (accented, "foreigner reading Vietnamese"), unless the client\n    # asked for a CJK language explicitly.\n    if (raw_lang.startswith("vi") or VI_CHARS.search(text)) and lang in ("auto", "en"):\n        text, lang, note = latinize_vi(text), "en", "vi->latin/en"\n    fmt = "ogg" if str(p.get("format") or p.get("media_type") or "wav").lower() == "ogg" else "wav"\n    try: speed = min(2.0, max(0.5, float(p.get("speed", 1.0))))\n    except Exception: speed = 1.0\n    try: seed = int(p.get("seed", -1))\n    except Exception: seed = -1\n    STATS["requests"] += 1\n    try:\n        data, sr, dur, ms = await run_in_threadpool(synth, text, voice, lang, speed, fmt, seed)\n    except Exception as e:\n        STATS["errors"] += 1; traceback.print_exc()\n        if "non-empty list" in str(e):\n            return JSONResponse({"ok": False, "error": f"no speakable text for lang \'{lang}\' (GPT-SoVITS supports {\', \'.join(LANGS)})"}, 422)\n        return JSONResponse({"ok": False, "error": f"{type(e).__name__}: {e}"}, 500)\n    STATS["last_ms"] = ms\n    print(f"[rp-tts] {voice}/{lang}{\' \'+note if note else \'\'} {len(text)} chars -> {dur:.1f}s audio in {ms} ms", flush=True)\n    return Response(data, media_type="audio/ogg" if fmt == "ogg" else "audio/wav",\n                    headers={"X-Voice": voice, "X-Lang": lang + (f" ({note})" if note else ""), "X-Duration": f"{dur:.2f}", "X-Gen-Ms": str(ms),\n                             "Cache-Control": "no-store"})\n\nif __name__ == "__main__":\n    uvicorn.run(app, host=args.host, port=args.port, log_level="warning")\n')

def pick(files, prefer):
    for p in prefer:
        for f in files:
            if os.path.basename(f) == p: return f
    return max(files, key=os.path.getmtime) if files else None
os.makedirs("models", exist_ok=True); os.makedirs("/content/rp_voices", exist_ok=True)
gpt = pick(glob.glob(f"{W}/*.ckpt"), ["vita-e15.ckpt"]); sov = pick(glob.glob(f"{W}/*.pth"), ["vita_e15_s390.pth"])
assert gpt and sov, "❌ Thiếu file .ckpt (GPT) hoặc .pth (SoVITS) trong " + W
gpt = shutil.copy(gpt, "models/"); sov = shutil.copy(sov, "models/")
print("🧠 GPT:", gpt, "| SoVITS:", sov)

def dur(p):
    r = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",p], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0
def prep_audio(src, vid):
    dst = f"/content/rp_voices/{vid}.wav"; d = dur(src)
    if d < 3: return None, d, False
    cut = d > 9.8   # GPT-SoVITS chỉ nhận audio mẫu 3-10 giây
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",src] + (["-t","9.0"] if cut else []) + ["-ac","1","-ar","32000",dst], check=True)
    return dst, min(d, 9.0), cut

LISTS = None
def from_lists(src):
    global LISTS
    if LISTS is None:
        LISTS = []
        for pat in [f"{VOICE_DIR}/**/*.list", "/content/drive/MyDrive/GPT-SoVITS/logs/*/*.list",
                    "/content/drive/MyDrive/GPT-SoVITS/output/*/*.list", "/content/drive/MyDrive/GPT-SoVITS/output/*.list"]:
            for f in glob.glob(pat, recursive=True):
                try: LISTS += [l.rstrip("\n").split("|") for l in open(f, encoding="utf-8", errors="ignore") if l.count("|") >= 3]
                except Exception: pass
    b = os.path.basename(src)
    for parts in LISTS:
        if os.path.basename(parts[0]) == b: return parts[3].strip(), parts[2].strip().lower()
    return None, None
def sidecar(src):
    stem = os.path.splitext(src)[0]
    for ext in (".txt", ".lab"):
        if os.path.exists(stem + ext): return open(stem + ext, encoding="utf-8").read().strip()
    return None
WHISPER_PY = """
import sys
from faster_whisper import WhisperModel
path, dev = sys.argv[1], sys.argv[2]
try:
    m = WhisperModel("large-v3-turbo", device=dev, compute_type="float16" if dev == "cuda" else "int8")
    segs, info = m.transcribe(path, beam_size=5)
except Exception as e:
    print("cuda failed, using cpu:", e, file=sys.stderr)
    m = WhisperModel("small", device="cpu", compute_type="int8")
    segs, info = m.transcribe(path, beam_size=5)
print(info.language + "|||" + "".join(s.text for s in segs).strip())
"""
def whisper(path):
    open("/content/rp_whisper.py", "w").write(WHISPER_PY)
    r = subprocess.run([PY, "/content/rp_whisper.py", path, "cuda" if GPU else "cpu"], capture_output=True, text=True)
    out = r.stdout.strip().splitlines()
    if not out or "|||" not in out[-1]: print("   (Whisper lỗi)", r.stderr[-400:]); return None, None
    lang, text = out[-1].split("|||", 1); return text.strip(), lang.strip()

LANG_OK = {"ja","en","zh","ko","yue"}
VOICES = {}
def add_voice(vid, src, text=None, lang=None, gpt_=None, sov_=None, name=None):
    path, d, cut = prep_audio(src, vid)
    if not path: print(f"⏭️ Bỏ {os.path.basename(src)} (ngắn hơn 3 giây)"); return
    how = "nhập tay" if text else ""
    if not text and not cut:
        text = sidecar(src); how = "file .txt" if text else ""
        if not text:
            t, l = from_lists(src)
            if t: text, lang, how = t, (lang or l), "file .list lúc train"
    if not text:
        t, l = whisper(path)
        if t: text, lang, how = t, l, "Whisper tự nghe" + (" (đã cắt còn 9 giây)" if cut else "")
    if not lang and text:  # đoán ngôn ngữ theo chữ viết
        lang = "ja" if re.search(r"[\u3040-\u30ff]", text) else "ko" if re.search(r"[\uac00-\ud7af]", text) else "zh" if re.search(r"[\u4e00-\u9fff]", text) else "en"
    lang = (lang or "ja").lower(); lang = {"jp":"ja","cn":"zh"}.get(lang, lang)
    if lang not in LANG_OK: print(f"   ⚠️ ngôn ngữ audio mẫu '{lang}' không được hỗ trợ → dùng chế độ không cần lời mẫu"); text = ""; lang = "ja"
    VOICES[vid] = {"name": name or vid, "gpt": gpt_ or gpt, "sovits": sov_ or sov, "ref": path, "text": text or "", "lang": lang}
    print(f"🎤 {vid}: {os.path.basename(src)} ({d:.1f}s) · lang={lang} · lời mẫu [{how or 'không có → chế độ không lời mẫu'}]: {text or '—'}")

files = [f for f in glob.glob(f"{VOICE_DIR}/*") if f.lower().endswith((".wav",".mp3",".flac",".ogg",".m4a"))]
main = next((f for f in files if os.path.basename(f) == "ref.wav"), None) or next((f for f in files if "clean" in f), None) or (files[0] if files else None)
assert main, f"❌ Không có audio mẫu (ref.wav) trong {VOICE_DIR}"
add_voice("vita", main, REF_TEXT.strip() or None, REF_LANG if REF_TEXT.strip() else None, name="Vita")
for f in files:
    if f == main: continue
    vid = "vita_" + re.sub(r"[^a-z0-9]+", "_", os.path.splitext(os.path.basename(f))[0].lower().replace("vita_", "")).strip("_")
    add_voice(vid, f, name="Vita · " + vid[5:])
if EXTRA_VOICES_DIR and os.path.isdir(EXTRA_VOICES_DIR):
    for d in sorted(glob.glob(f"{EXTRA_VOICES_DIR}/*/")):
        vid = re.sub(r"[^a-z0-9_]+", "_", os.path.basename(d.rstrip("/")).lower())
        au = [f for f in glob.glob(d + "*") if f.lower().endswith((".wav",".mp3",".flac",".ogg",".m4a"))]
        if not au: continue
        t = open(d + "ref.txt", encoding="utf-8").read().strip() if os.path.exists(d + "ref.txt") else None
        l = open(d + "lang.txt").read().strip() if os.path.exists(d + "lang.txt") else None
        g = glob.glob(d + "*.ckpt"); s = glob.glob(d + "*.pth")
        add_voice(vid, au[0], t, l, shutil.copy(g[0], "models/") if g else None, shutil.copy(s[0], "models/") if s else None)
json.dump({"version": "v2Pro", "default": "vita", "voices": VOICES}, open("/content/voices.json", "w"), ensure_ascii=False, indent=1)

if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True)
subprocess.run("pkill -f rp_tts_server.py; pkill -f inference_webui; pkill -f cloudflared", shell=True); time.sleep(1)
def start_server():
    return subprocess.Popen(f"cd /content/GPT-SoVITS && MPLBACKEND=Agg {PY} rp_tts_server.py --voices /content/voices.json --port 9880 > /content/tts.log 2>&1", shell=True, executable="/bin/bash")
def start_tunnel():
    open("/content/cf.log", "w").close()
    return subprocess.Popen("/content/cloudflared tunnel --protocol http2 --no-autoupdate --url http://127.0.0.1:9880 > /content/cf.log 2>&1", shell=True)
def get(url, timeout=10, data=None):
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json", "Origin": "https://hoangnguyencccc194-blip.github.io"})
    with urllib.request.urlopen(req, timeout=timeout) as r: return r.status, r.read(), dict(r.headers)
srv = start_server(); print("⏳ Đang nạp model (1-3 phút)...", flush=True)
t0 = time.time()
while True:
    if srv.poll() is not None: print("❌ Máy chủ dừng. Log:"); print(open("/content/tts.log").read()[-4000:]); raise SystemExit
    try:
        st, body, _ = get("http://127.0.0.1:9880/health", 3)
        if st == 200: break
    except Exception: pass
    if time.time() - t0 > 900: print("❌ Quá 15 phút chưa lên. Log:"); print(open("/content/tts.log").read()[-4000:]); raise SystemExit
    time.sleep(3)
print("✅ Máy chủ đã chạy:", json.loads(body)["voices"])
try:  # chạy thử 1 câu để làm nóng GPU
    st, wav, h = get("http://127.0.0.1:9880/tts", 300, json.dumps({"text": "こんにちは、マネージャー。", "lang": "ja"}).encode())
    print(f"🔊 Thử đọc: {len(wav)//1024} KB, {h.get('x-duration') or h.get('X-Duration')} giây audio, tạo trong {h.get('x-gen-ms') or h.get('X-Gen-Ms')} ms")
    from IPython.display import Audio, display; display(Audio(wav, autoplay=False))
except Exception as e: print("⚠️ Thử đọc lỗi:", e); print(open("/content/tts.log").read()[-2000:])
cf = start_tunnel(); URL = None
def find_url():
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read()); return m.group(0) if m else None
def announce(u):
    from IPython.display import display, HTML
    link = APP_URL + "#tts=" + u
    print("\n" + "=" * 70 + f"\n🌐 LINK MÁY CHỦ TTS:  {u}\n📱 MỞ RP CHAT (tự điền link):  {link}\n" + "=" * 70, flush=True)
    display(HTML(f'<div style="font-size:16px;line-height:1.8">🌐 Máy chủ TTS: <code>{u}</code><br>📱 <a href="{link}" target="_blank"><b>Bấm để mở RP Chat với giọng vita</b></a></div>'))
for _ in range(60):
    URL = find_url()
    if URL: break
    time.sleep(2)
ok = False
for _ in range(30):  # chờ DNS của trycloudflare
    try: ok = get(URL + "/health", 10)[0] == 200
    except Exception: ok = False
    if ok: break
    time.sleep(3)
print("✅ Link công khai kiểm tra /health: OK" if ok else "⚠️ Link chưa trả lời /health (đợi thêm 1 phút rồi bấm Kiểm tra trong app)")
announce(URL)
seen = 0
try:
    while True:
        time.sleep(5)
        log = open("/content/tts.log", errors="ignore").read()
        for line in log[seen:].splitlines():
            if line.startswith("[rp-tts]") or "Error" in line: print(line, flush=True)
        seen = len(log)
        if srv.poll() is not None:
            print("❌ Máy chủ dừng, khởi động lại...", flush=True); srv = start_server(); seen = 0
        if cf.poll() is not None:
            print("⚠️ Tunnel rớt, mở lại (link sẽ đổi)...", flush=True); cf = start_tunnel(); time.sleep(10)
            u = find_url()
            if u: URL = u; announce(URL)
except KeyboardInterrupt:
    print("Đã dừng theo dõi (máy chủ vẫn chạy nền). Muốn tắt hẳn: Runtime → Disconnect and delete runtime.")


## Dùng trong RP Chat
1. Bấm link 📱 ở trên (hoặc copy link 🌐 → RP Chat › ⚙️ Cài đặt › **Giọng nói (GPT-SoVITS)** › dán vào ô *Link máy chủ TTS* › **Kiểm tra**).
2. Vào chat: bấm 🔊 dưới tin nhắn để nghe, hoặc bật **Tự đọc câu trả lời**. Nút 🎤 cạnh ô nhập để nói thay vì gõ; bấm giữ 🎤 (hoặc menu ⋯ › *Trò chuyện rảnh tay*) để nói chuyện liên tục.
3. Mỗi nhân vật chọn giọng ở **Sửa nhân vật › Giọng nói**.

**Lỗi thường gặp:** link trycloudflare chết khi Colab ngắt (giới hạn GPU miễn phí) → chạy lại ô 3, link mới sẽ hiện. App tự chuyển sang giọng trình duyệt khi không gọi được máy chủ.